# Xây dựng Dataset dạng Token ID cho Mô hình Vietnamese Transformer Summarization

Notebook này thực hiện mã hóa (tokenize) tập dữ liệu tóm tắt tiếng Việt bằng **SentencePiece Tokenizer** và chuyển đổi sang dạng Token ID phù hợp cho mô hình Transformer Encoder-Decoder (ví dụ: BART, T5).

Quy trình được thực hiện theo đúng hướng dẫn thiết kế trong tài liệu `huong_dan_build_dataset_dang_token_id.pdf`.

In [1]:
# Cài đặt thư viện cần thiết nếu chạy trên Kaggle (bỏ dấu comment nếu cần)
!pip install -q sentencepiece torch tqdm pandas numpy

## 1. Cấu hình Đường dẫn & Tham số

**Lưu ý**: Hãy điền đường dẫn thực tế của các file trên Kaggle. Mặc định, nếu để trống, notebook sẽ tự động tìm kiếm đường dẫn cục bộ (nếu có).

In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import sentencepiece as spm
import torch
from torch.utils.data import Dataset, DataLoader

# HÃY ĐIỀN ĐƯỜNG DẪN THỰC TẾ TRÊN KAGGLE Ở ĐÂY
TRAIN_JSONL_PATH = "/kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/train.jsonl"
VALID_JSONL_PATH = "/kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/valid.jsonl"
TEST_JSONL_PATH = "/kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/test.jsonl"
TOKENIZER_MODEL_PATH = "/kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/vietnamese_spm.model"
OUTPUT_DIR = "/kaggle/working/processed_dataset"

# Ví dụ đường dẫn trên Kaggle:
# TRAIN_JSONL_PATH = "/kaggle/input/vietnamese-summarization/train.jsonl"
# VALID_JSONL_PATH = "/kaggle/input/vietnamese-summarization/valid.jsonl"
# TEST_JSONL_PATH = "/kaggle/input/vietnamese-summarization/test.jsonl"
# TOKENIZER_MODEL_PATH = "/kaggle/input/vietnamese-summarization-spm/vietnamese_spm.model"
# OUTPUT_DIR = "/kaggle/working/processed_dataset"

# Tự động gán đường dẫn cục bộ nếu để trống (tiện lợi khi chạy thử nghiệm)
if not TRAIN_JSONL_PATH:
    local_path = "../data/processed/train.jsonl"
    if os.path.exists(local_path):
        TRAIN_JSONL_PATH = local_path
        print(f"Tự động sử dụng đường dẫn Train cục bộ: {TRAIN_JSONL_PATH}")
    else:
        print("CẢNH BÁO: TRAIN_JSONL_PATH đang để trống. Hãy điền đường dẫn trước khi chạy.")

if not VALID_JSONL_PATH:
    local_path = "../data/processed/valid.jsonl"
    if os.path.exists(local_path):
        VALID_JSONL_PATH = local_path
        print(f"Tự động sử dụng đường dẫn Valid cục bộ: {VALID_JSONL_PATH}")
    else:
        print("CẢNH BÁO: VALID_JSONL_PATH đang để trống. Hãy điền đường dẫn trước khi chạy.")

if not TEST_JSONL_PATH:
    local_path = "../data/processed/test.jsonl"
    if os.path.exists(local_path):
        TEST_JSONL_PATH = local_path
        print(f"Tự động sử dụng đường dẫn Test cục bộ: {TEST_JSONL_PATH}")
    else:
        print("CẢNH BÁO: TEST_JSONL_PATH đang để trống. Hãy điền đường dẫn trước khi chạy.")

if not TOKENIZER_MODEL_PATH:
    local_path = "../data/tokenizer/model/vietnamese_spm.model"
    if os.path.exists(local_path):
        TOKENIZER_MODEL_PATH = local_path
        print(f"Tự động sử dụng đường dẫn Model cục bộ: {TOKENIZER_MODEL_PATH}")
    else:
        print("CẢNH BÁO: TOKENIZER_MODEL_PATH đang để trống. Hãy điền đường dẫn trước khi chạy.")

if not OUTPUT_DIR:
    OUTPUT_DIR = "./processed_dataset"
    print(f"Tự động thiết lập thư mục đầu ra: {OUTPUT_DIR}")

print("\n--- CẤU HÌNH ĐƯỜNG DẪN ---")
print(f"Train file:     {TRAIN_JSONL_PATH}")
print(f"Valid file:     {VALID_JSONL_PATH}")
print(f"Test file:      {TEST_JSONL_PATH}")
print(f"Model file:     {TOKENIZER_MODEL_PATH}")
print(f"Output dir:     {OUTPUT_DIR}")


--- CẤU HÌNH ĐƯỜNG DẪN ---
Train file:     /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/train.jsonl
Valid file:     /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/valid.jsonl
Test file:      /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/test.jsonl
Model file:     /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/vietnamese_spm.model
Output dir:     /kaggle/working/processed_dataset


## 2. Siêu tham số & ID Token đặc biệt

Định nghĩa các tham số độ dài tối đa và các ID token đặc biệt cố định:
- `max_source_length = 768` (độ dài tối đa của encoder input, bao gồm cả `<eos>`)
- `max_target_length = 96` (độ dài tối đa của decoder input và labels, bao gồm cả `<bos>` hoặc `<eos>`)
- ID đặc biệt: `<pad> = 0`, `<unk> = 1`, `<bos> = 2`, `<eos> = 3`

In [3]:
MAX_SOURCE_LENGTH = 768
MAX_TARGET_LENGTH = 96

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

## 3. Nạp và Kiểm tra Tokenizer

Nạp mô hình SentencePiece và kiểm tra xem các ID của các token đặc biệt có trùng khớp với thiết kế cố định hay không.

In [4]:
if TOKENIZER_MODEL_PATH and os.path.exists(TOKENIZER_MODEL_PATH):
    print(f"Đang nạp SentencePiece Tokenizer từ: {TOKENIZER_MODEL_PATH}")
    sp = spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL_PATH)
    
    # Kiểm tra tính khớp của các ID đặc biệt
    print(f"Kích thước bộ từ vựng (Vocab size): {sp.get_piece_size()}")
    print(f"  - PAD ID: {sp.pad_id()} (Kỳ vọng: {PAD_ID})")
    print(f"  - UNK ID: {sp.unk_id()} (Kỳ vọng: {UNK_ID})")
    print(f"  - BOS ID: {sp.bos_id()} (Kỳ vọng: {BOS_ID})")
    print(f"  - EOS ID: {sp.eos_id()} (Kỳ vọng: {EOS_ID})")
    
    assert sp.pad_id() == PAD_ID, f"LỖI: PAD ID của model ({sp.pad_id()}) khác với cấu hình ({PAD_ID})!"
    assert sp.unk_id() == UNK_ID, f"LỖI: UNK ID của model ({sp.unk_id()}) khác với cấu hình ({UNK_ID})!"
    assert sp.bos_id() == BOS_ID, f"LỖI: BOS ID của model ({sp.bos_id()}) khác với cấu hình ({BOS_ID})!"
    assert sp.eos_id() == EOS_ID, f"LỖI: EOS ID của model ({sp.eos_id()}) khác với cấu hình ({EOS_ID})!"
    print("\n-> Xác thực Tokenizer thành công! Các ID đặc biệt trùng khớp hoàn toàn.")
else:
    print("LỖI: Không tìm thấy model file. Vui lòng kiểm tra lại TOKENIZER_MODEL_PATH.")

Đang nạp SentencePiece Tokenizer từ: /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/vietnamese_spm.model
Kích thước bộ từ vựng (Vocab size): 16000
  - PAD ID: 0 (Kỳ vọng: 0)
  - UNK ID: 1 (Kỳ vọng: 1)
  - BOS ID: 2 (Kỳ vọng: 2)
  - EOS ID: 3 (Kỳ vọng: 3)

-> Xác thực Tokenizer thành công! Các ID đặc biệt trùng khớp hoàn toàn.


## 4. Quy trình Xử lý và Mã hóa (Tokenization Pipeline)

Hàm `process_jsonl` thực hiện quy trình sau:
1. Đọc từng dòng của file JSONL gốc, lấy văn bản `Contents` và `Summary` (tự động ánh xạ từ `article`/`summary` nếu có).
2. Mã hóa `Contents` thành `source_core_ids` (chưa có `<eos>`).
   - Nếu độ dài `source_core_ids > 767`: tiến hành cắt ngắn (truncate) còn `767` token rồi thêm `<eos>` ở cuối (tổng là 768).
   - Ngược lại, giữ nguyên và thêm `<eos>` ở cuối.
   - Thiết lập flag `was_source_truncated = True` nếu bị cắt ngắn.
3. Mã hóa `Summary` thành `target_core_ids` (chưa có `<bos>` / `<eos>`).
   - Nếu độ dài `target_core_ids > 95`: **loại bỏ (drop)** mẫu này (vì target sau khi thêm BOS và EOS sẽ có độ dài tối đa là 96).
4. Tạo `decoder_input_ids` bằng cách thêm `<bos>` vào đầu: `[bos_id] + target_core_ids`.
5. Tạo `labels` bằng cách thêm `<eos>` vào cuối: `target_core_ids + [eos_id]`.
6. Ghi mẫu kết quả xuống đĩa dạng JSONL với cấu trúc:
```json
{
  "id": "...",
  "source_ids": [...],
  "decoder_input_ids": [...],
  "labels": [...],
  "source_len": ...,
  "target_len": ...,
  "was_source_truncated": true/false
}
```

In [5]:
def process_jsonl(input_path, output_path, sp_model, split_name="data"):
    if not input_path or not os.path.exists(input_path):
        print(f"Cảnh báo: Đường dẫn '{input_path}' không tồn tại. Bỏ qua split này.")
        return None
        
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    total_samples = 0
    dropped_samples = 0
    truncated_sources = 0
    saved_samples = 0
    
    # Lưu thống kê độ dài để vẽ biểu đồ và phân tích sau đó
    stats = []
    
    print(f"\n--- Bắt đầu xử lý tập [{split_name}] ---")
    print(f"Đọc file nguồn: {input_path}")
    print(f"Ghi file đích:   {output_path}")
    
    # Tính số dòng để hiển trình trực quan
    with open(input_path, 'r', encoding='utf-8') as f:
        num_lines = sum(1 for _ in f)
        
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:
         
        for idx, line in enumerate(tqdm(fin, total=num_lines, desc=f"Tokenizing {split_name}")):
            total_samples += 1
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                print(f"Lỗi định dạng JSON ở dòng {idx+1}. Bỏ qua.")
                continue
                
            # Lấy trường Contents và Summary (hỗ trợ cả article/summary)
            contents = data.get("Contents") or data.get("article")
            summary = data.get("Summary") or data.get("summary")
            
            if not contents or not summary:
                print(f"Cảnh báo dòng {idx+1}: Thiếu Contents hoặc Summary. Bỏ qua.")
                continue
                
            # Gán ID duy nhất cho mỗi mẫu
            sample_id = data.get("id") or f"{split_name}_{idx+1}"
            
            # 1. Encode Source (Contents)
            source_core_ids = sp_model.encode_as_ids(contents)
            source_core_len = len(source_core_ids)
            
            was_source_truncated = False
            if source_core_len > (MAX_SOURCE_LENGTH - 1):
                # Cắt ngắn còn 767 token và nối thêm <eos> (ID 3)
                source_ids = source_core_ids[:MAX_SOURCE_LENGTH - 1] + [EOS_ID]
                was_source_truncated = True
                truncated_sources += 1
            else:
                source_ids = source_core_ids + [EOS_ID]
                
            # 2. Encode Target (Summary)
            target_core_ids = sp_model.encode_as_ids(summary)
            target_core_len = len(target_core_ids)
            
            # Lọc bỏ mẫu nếu Summary sau encode dài hơn 95 token
            if target_core_len > (MAX_TARGET_LENGTH - 1):
                dropped_samples += 1
                continue
                
            # 3. Tạo decoder_input_ids và labels
            decoder_input_ids = [BOS_ID] + target_core_ids
            labels = target_core_ids + [EOS_ID]
            
            # Ràng buộc kiểm tra tính hợp lệ
            assert len(decoder_input_ids) == len(labels)
            assert len(decoder_input_ids) <= MAX_TARGET_LENGTH
            
            # Khởi tạo dict chứa token id của mẫu
            output_sample = {
                "id": sample_id,
                "source_ids": source_ids,
                "decoder_input_ids": decoder_input_ids,
                "labels": labels,
                "source_len": len(source_ids),
                "target_len": len(labels),
                "was_source_truncated": was_source_truncated
            }
            
            # Ghi trực tiếp ra file JSONL
            fout.write(json.dumps(output_sample, ensure_ascii=False) + "\n")
            
            # Thu thập stats
            stats.append({
                "source_len": len(source_ids),
                "target_len": len(labels),
                "was_source_truncated": was_source_truncated
            })
            saved_samples += 1
            
    # Hiển thị thống kê cho split
    print(f"-> Kết quả xử lý tập [{split_name}]:")
    print(f"   - Tổng số mẫu ban đầu:              {total_samples:,}")
    print(f"   - Số mẫu bị loại bỏ (Target > 95):  {dropped_samples:,} ({dropped_samples/total_samples*100:.2f}%)")
    print(f"   - Số mẫu bị truncate ở Source (>767): {truncated_sources:,} ({truncated_sources/total_samples*100:.2f}%)")
    print(f"   - Số mẫu lưu thành công:            {saved_samples:,} ({saved_samples/total_samples*100:.2f}%)")
    
    return pd.DataFrame(stats)

## 5. Tiến hành Build Dataset và Lưu Config

Chạy hàm build dataset cho cả 3 tập dữ liệu: Train, Validation và Test. Đồng thời, lưu trữ cấu hình tokenizer config vào file JSON như thiết kế.

In [6]:
# Xác định các file đầu ra tương ứng
train_output = os.path.join(OUTPUT_DIR, "train_token_id.jsonl")
valid_output = os.path.join(OUTPUT_DIR, "valid_token_id.jsonl")
test_output = os.path.join(OUTPUT_DIR, "test_token_id.jsonl")

# Tiến hành xử lý và mã hóa
train_stats = process_jsonl(TRAIN_JSONL_PATH, train_output, sp, split_name="train")
valid_stats = process_jsonl(VALID_JSONL_PATH, valid_output, sp, split_name="valid")
test_stats = process_jsonl(TEST_JSONL_PATH, test_output, sp, split_name="test")

# Lưu file tokenizer_config.json theo đúng hướng dẫn
config_info = {
    "tokenizer_path": TOKENIZER_MODEL_PATH,
    "pad_id": PAD_ID,
    "unk_id": UNK_ID,
    "bos_id": BOS_ID,
    "eos_id": EOS_ID,
    "vocab_size": sp.get_piece_size() if 'sp' in locals() else 16000,
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "source_policy": "truncate_keep_eos",
    "target_policy": "drop_if_len_greater_than_95",
    "padding_policy": "dynamic_padding_in_collate_fn"
}

config_output_path = os.path.join(OUTPUT_DIR, "tokenizer_config.json")
with open(config_output_path, 'w', encoding='utf-8') as f:
    json.dump(config_info, f, indent=4, ensure_ascii=False)
print(f"\n-> Đã lưu cấu hình Tokenizer Config vào: {config_output_path}")


--- Bắt đầu xử lý tập [train] ---
Đọc file nguồn: /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/train.jsonl
Ghi file đích:   /kaggle/working/processed_dataset/train_token_id.jsonl


Tokenizing train:   0%|          | 0/197580 [00:00<?, ?it/s]

-> Kết quả xử lý tập [train]:
   - Tổng số mẫu ban đầu:              197,580
   - Số mẫu bị loại bỏ (Target > 95):  3,739 (1.89%)
   - Số mẫu bị truncate ở Source (>767): 33,102 (16.75%)
   - Số mẫu lưu thành công:            193,841 (98.11%)

--- Bắt đầu xử lý tập [valid] ---
Đọc file nguồn: /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/valid.jsonl
Ghi file đích:   /kaggle/working/processed_dataset/valid_token_id.jsonl


Tokenizing valid:   0%|          | 0/24697 [00:00<?, ?it/s]

-> Kết quả xử lý tập [valid]:
   - Tổng số mẫu ban đầu:              24,697
   - Số mẫu bị loại bỏ (Target > 95):  453 (1.83%)
   - Số mẫu bị truncate ở Source (>767): 4,154 (16.82%)
   - Số mẫu lưu thành công:            24,244 (98.17%)

--- Bắt đầu xử lý tập [test] ---
Đọc file nguồn: /kaggle/input/datasets/tranducthinh2006/du-lieu-de-build-dataset-token-id/test.jsonl
Ghi file đích:   /kaggle/working/processed_dataset/test_token_id.jsonl


Tokenizing test:   0%|          | 0/24698 [00:00<?, ?it/s]

-> Kết quả xử lý tập [test]:
   - Tổng số mẫu ban đầu:              24,698
   - Số mẫu bị loại bỏ (Target > 95):  456 (1.85%)
   - Số mẫu bị truncate ở Source (>767): 4,169 (16.88%)
   - Số mẫu lưu thành công:            24,242 (98.15%)

-> Đã lưu cấu hình Tokenizer Config vào: /kaggle/working/processed_dataset/tokenizer_config.json


## 6. Phân tích Thống kê sau khi Build Dataset

Kiểm tra xem độ dài tối đa của Source và Target có thỏa mãn các kỳ vọng hay không:
- `source_len` cực đại phải `<= 768`
- `target_len` cực đại phải `<= 96`
- Không tồn tại mẫu rỗng, không có labels toàn pad.

In [7]:
def analyze_stats(df_stats, split_name):
    if df_stats is None or df_stats.empty:
        print(f"Không có dữ liệu thống kê cho tập [{split_name}].")
        return
        
    print(f"\n=== PHÂN TÍCH THỐNG KÊ TẬP [{split_name.upper()}] ===")
    print(f"Số lượng mẫu còn lại sau khi lọc: {len(df_stats):,}")
    
    # Thống kê Source Length
    src_max = df_stats["source_len"].max()
    src_min = df_stats["source_len"].min()
    src_mean = df_stats["source_len"].mean()
    src_median = df_stats["source_len"].median()
    
    # Thống kê Target Length
    tgt_max = df_stats["target_len"].max()
    tgt_min = df_stats["target_len"].min()
    tgt_mean = df_stats["target_len"].mean()
    tgt_median = df_stats["target_len"].median()
    
    print(f"  - Source Length: Min={src_min} | Max={src_max} (Kỳ vọng <= 768) | Mean={src_mean:.2f} | Median={src_median:.1f}")
    print(f"  - Target Length: Min={tgt_min} | Max={tgt_max} (Kỳ vọng <= 96) | Mean={tgt_mean:.2f} | Median={tgt_median:.1f}")
    
    # Assertion kiểm tra tự động
    assert src_max <= MAX_SOURCE_LENGTH, f"Lỗi: Phát hiện source_len {src_max} vượt quá {MAX_SOURCE_LENGTH}!"
    assert tgt_max <= MAX_TARGET_LENGTH, f"Lỗi: Phát hiện target_len {tgt_max} vượt quá {MAX_TARGET_LENGTH}!"
    assert src_min > 0, "Lỗi: Phát hiện mẫu rỗng (độ dài 0) ở Source!"
    assert tgt_min > 0, "Lỗi: Phát hiện mẫu rỗng (độ dài 0) ở Target!"
    print("-> Xác nhận: Các ràng buộc về phân phối độ dài đều hợp lệ.")

if train_stats is not None:
    analyze_stats(train_stats, "train")
if valid_stats is not None:
    analyze_stats(valid_stats, "valid")
if test_stats is not None:
    analyze_stats(test_stats, "test")


=== PHÂN TÍCH THỐNG KÊ TẬP [TRAIN] ===
Số lượng mẫu còn lại sau khi lọc: 193,841
  - Source Length: Min=106 | Max=768 (Kỳ vọng <= 768) | Mean=499.62 | Median=490.0
  - Target Length: Min=12 | Max=96 (Kỳ vọng <= 96) | Mean=46.39 | Median=44.0
-> Xác nhận: Các ràng buộc về phân phối độ dài đều hợp lệ.

=== PHÂN TÍCH THỐNG KÊ TẬP [VALID] ===
Số lượng mẫu còn lại sau khi lọc: 24,244
  - Source Length: Min=112 | Max=768 (Kỳ vọng <= 768) | Mean=500.13 | Median=492.0
  - Target Length: Min=14 | Max=96 (Kỳ vọng <= 96) | Mean=46.30 | Median=44.0
-> Xác nhận: Các ràng buộc về phân phối độ dài đều hợp lệ.

=== PHÂN TÍCH THỐNG KÊ TẬP [TEST] ===
Số lượng mẫu còn lại sau khi lọc: 24,242
  - Source Length: Min=108 | Max=768 (Kỳ vọng <= 768) | Mean=499.65 | Median=489.0
  - Target Length: Min=12 | Max=96 (Kỳ vọng <= 96) | Mean=46.39 | Median=44.0
-> Xác nhận: Các ràng buộc về phân phối độ dài đều hợp lệ.


## 7. Kiểm tra Giải mã Ngược (Decode Verification)

Thực hiện chọn ngẫu nhiên một mẫu từ tập Validation mới được build, giải mã ngược từ dạng ID về dạng Text gốc sử dụng SentencePiece. 

Kỳ vọng:
- `source_ids` giải mã ngược gần giống `Contents` gốc (có thể bị cắt ngắn).
- `decoder_input_ids` giải mã ngược giống hệt `Summary` gốc.
- `labels` giải mã ngược giống hệt `Summary` gốc.
- Sự khác biệt: `decoder_input_ids` có `<bos>` ở đầu, `labels` có `<eos>` ở cuối.

In [8]:
def verify_random_sample(output_jsonl_path, sp_model):
    if not os.path.exists(output_jsonl_path):
        print(f"Không tìm thấy file kết quả tại: {output_jsonl_path}")
        return
        
    # Đọc tối đa 1000 mẫu đầu tiên để chọn ngẫu nhiên nhanh chóng
    samples = []
    with open(output_jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            samples.append(json.loads(line))
            if len(samples) >= 1000:
                break
                
    if not samples:
        print("Không có mẫu dữ liệu nào để kiểm tra.")
        return
        
    sample = random.choice(samples)
    print("\n" + "="*80)
    print(f"   KIỂM TRA GIẢI MÃ NGƯỢC (DECODE VERIFICATION) - SAMPLE ID: {sample['id']}")
    print("="*80)
    
    source_ids = sample["source_ids"]
    decoder_input_ids = sample["decoder_input_ids"]
    labels = sample["labels"]
    
    # Bỏ qua các special token ID trước khi giải mã ngược để so sánh
    src_ids_to_decode = [tok for tok in source_ids if tok not in [PAD_ID, EOS_ID]]
    dec_in_ids_to_decode = [tok for tok in decoder_input_ids if tok not in [PAD_ID, BOS_ID]]
    lbl_ids_to_decode = [tok for tok in labels if tok not in [PAD_ID, EOS_ID]]
    
    src_decoded = sp_model.decode(src_ids_to_decode)
    dec_in_decoded = sp_model.decode(dec_in_ids_to_decode)
    lbl_decoded = sp_model.decode(lbl_ids_to_decode)
    
    print(f"1. [Source (Contents)]\n   - Độ dài: Gốc={len(source_ids)}, Bỏ special={len(src_ids_to_decode)}")
    print(f"   - Văn bản sau giải mã:\n   {src_decoded[:350]}...")
    print(f"   - Truncated? {sample['was_source_truncated']}\n")
    
    print(f"2. [Decoder Input]\n   - Độ dài: Gốc={len(decoder_input_ids)}, Bỏ special={len(dec_in_ids_to_decode)}")
    print(f"   - Token đầu tiên: ID={decoder_input_ids[0]} (Kỳ vọng BOS={BOS_ID})")
    print(f"   - Văn bản sau giải mã:\n   {dec_in_decoded}\n")
    
    print(f"3. [Labels (Target)]\n   - Độ dài: Gốc={len(labels)}, Bỏ special={len(lbl_ids_to_decode)}")
    print(f"   - Token cuối cùng: ID={labels[-1]} (Kỳ vọng EOS={EOS_ID})")
    print(f"   - Văn bản sau giải mã:\n   {lbl_decoded}\n")
    
    # Kiểm tra các logic shift và đặc điểm đặc biệt
    print("--- KẾT QUẢ KIỂM TRA LOGIC ---")
    print(f" * [Bắt đầu] decoder_input_ids bắt đầu bằng BOS (ID 2): {decoder_input_ids[0] == BOS_ID}")
    print(f" * [Kết thúc] labels kết thúc bằng EOS (ID 3):         {labels[-1] == EOS_ID}")
    print(f" * [Đồng bộ] decoder_input và labels giải mã giống nhau: {dec_in_decoded == lbl_decoded}")
    print(f" * [Độ dài] Hai chuỗi có cùng độ dài thật:            {len(decoder_input_ids) == len(labels)}")
    print("="*80)

valid_output_jsonl = os.path.join(OUTPUT_DIR, "valid_token_id.jsonl")
if 'sp' in locals() and os.path.exists(valid_output_jsonl):
    verify_random_sample(valid_output_jsonl, sp)


   KIỂM TRA GIẢI MÃ NGƯỢC (DECODE VERIFICATION) - SAMPLE ID: valid_966
1. [Source (Contents)]
   - Độ dài: Gốc=251, Bỏ special=250
   - Văn bản sau giải mã:
   Tại huyện Triệu Phong (tỉnh Quảng Trị), nhiều hộ gia đình trồng lúa hữu cơ cho ra loại gạo ngon, sạch, bán với giá từ 22.000 đồng đến 25.000 đồng/kg. Khi chưa xảy ra dịch COVID-19, gạo sạch này được ưa chuộng, bán ra các tỉnh phía Bắc và đi vào các tỉnh thành phía Nam. Vụ lúa hè thu 2021 này nông dân ở huyện Triệu Phong đã thu hoạch với sản lượng k...
   - Truncated? False

2. [Decoder Input]
   - Độ dài: Gốc=47, Bỏ special=46
   - Token đầu tiên: ID=2 (Kỳ vọng BOS=2)
   - Văn bản sau giải mã:
   Đã đến vụ thu hoạch, nhưng lúa từ vụ trước còn ứ lại hàng chục tấn. Để hỗ trợ nông dân, Liên đoàn Lao động huyện Triệu Phong (tỉnh Quảng Trị) kêu gọi người lao động vào cuộc giải cứu.

3. [Labels (Target)]
   - Độ dài: Gốc=47, Bỏ special=46
   - Token cuối cùng: ID=3 (Kỳ vọng EOS=3)
   - Văn bản sau giải mã:
   Đã đến vụ thu hoạch, như

## 8. Xây dựng PyTorch Dataset & collate_fn

Như tài liệu thiết kế chỉ ra, ta không đệm (padding) tĩnh các chuỗi Token ID khi lưu trữ để tối ưu hóa ổ cứng và RAM. Thay vào đó, ta thực hiện đệm động (dynamic padding) trong `collate_fn` của `DataLoader` khi gom các mẫu dữ liệu thành một batch.

### Hàm `summarization_collate_fn` thực hiện:
1. Tìm độ dài lớn nhất của `source_ids` và `labels` (cũng chính là độ dài của `decoder_input_ids`) trong batch hiện tại.
2. Sử dụng `PAD_ID = 0` để đệm các chuỗi ngắn hơn về độ dài lớn nhất của batch.
3. Tạo các mặt nạ (masks):
   - `source_padding_mask`: Kích thước `[batch_size, max_source_len]`. Các vị trí chứa token thực tế là `False` (cho phép attention nhìn), các vị trí chứa token đệm là `True` (ngăn attention nhìn).
   - `target_padding_mask`: Kích thước `[batch_size, max_target_len]`. Tương tự, `True` cho vị trí đệm và `False` cho token thực tế.
4. Gắn nhãn `labels` với `ignore_index = 0` (theo cấu hình loss ignore_index = pad_id = 0) để PyTorch `CrossEntropyLoss` không tính loss trên các token padding.

In [9]:
class SummarizationDataset(Dataset):
    def __init__(self, jsonl_path):
        self.samples = []
        print(f"Đang nạp dataset từ: {jsonl_path}")
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                self.samples.append(json.loads(line))
        print(f"-> Nạp thành công {len(self.samples):,} mẫu.")
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            "source_ids": sample["source_ids"],
            "decoder_input_ids": sample["decoder_input_ids"],
            "labels": sample["labels"]
        }

def summarization_collate_fn(batch):
    # Tách dữ liệu
    sources = [x["source_ids"] for x in batch]
    decoder_inputs = [x["decoder_input_ids"] for x in batch]
    labels_list = [x["labels"] for x in batch]
    
    # Tìm độ dài lớn nhất trong batch này
    max_src_len = max(len(s) for s in sources)
    max_tgt_len = max(len(d) for d in decoder_inputs)
    
    batch_size = len(batch)
    
    # Khởi tạo các tensors đệm với PAD_ID = 0
    padded_sources = torch.full((batch_size, max_src_len), PAD_ID, dtype=torch.long)
    padded_decoder_inputs = torch.full((batch_size, max_tgt_len), PAD_ID, dtype=torch.long)
    padded_labels = torch.full((batch_size, max_tgt_len), PAD_ID, dtype=torch.long)
    
    # Khởi tạo padding masks (True nghĩa là đệm - không được phép chú ý, False là thật)
    source_padding_mask = torch.ones((batch_size, max_src_len), dtype=torch.bool)
    target_padding_mask = torch.ones((batch_size, max_tgt_len), dtype=torch.bool)
    
    for i in range(batch_size):
        src_len = len(sources[i])
        tgt_len = len(decoder_inputs[i])
        
        # Đổ dữ liệu thật vào tensor
        padded_sources[i, :src_len] = torch.tensor(sources[i], dtype=torch.long)
        padded_decoder_inputs[i, :tgt_len] = torch.tensor(decoder_inputs[i], dtype=torch.long)
        padded_labels[i, :tgt_len] = torch.tensor(labels_list[i], dtype=torch.long)
        
        # Đánh dấu các token thật là False để Attention có thể chú ý
        source_padding_mask[i, :src_len] = False
        target_padding_mask[i, :tgt_len] = False
        
    return {
        "source_ids": padded_sources,
        "source_padding_mask": source_padding_mask,
        "decoder_input_ids": padded_decoder_inputs,
        "target_padding_mask": target_padding_mask,
        "labels": padded_labels
    }

## 9. Chạy Thử nghiệm DataLoader

Tiến hành nạp thử DataLoader để hiển thị shape và kiểm tra các giá trị thực tế sau đệm.

In [10]:
valid_output_jsonl = os.path.join(OUTPUT_DIR, "valid_token_id.jsonl")
if os.path.exists(valid_output_jsonl):
    dataset = SummarizationDataset(valid_output_jsonl)
    # Thiết lập batch_size nhỏ để dễ quan sát
    dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=summarization_collate_fn)
    
    # Lấy batch đầu tiên
    batch = next(iter(dataloader))
    
    print("\n" + "="*80)
    print("         THỬ NGHIỆM DATALOADER VÀ KIỂM TRA TENSOR ĐỘNG")
    print("="*80)
    print("1. Shape [source_ids]:        ", batch["source_ids"].shape)
    print("2. Shape [source_padding_mask]:", batch["source_padding_mask"].shape)
    print("3. Shape [decoder_input_ids]: ", batch["decoder_input_ids"].shape)
    print("4. Shape [target_padding_mask]:", batch["target_padding_mask"].shape)
    print("5. Shape [labels]:             ", batch["labels"].shape)
    
    # In ví dụ của mẫu đầu tiên trong batch
    print("\n* Chi tiết mẫu số 1 trong batch:")
    print(f"  - source_ids:\n    {batch['source_ids'][0].tolist()[:-1]}")
    print(f"  - source_padding_mask:\n    {batch['source_padding_mask'][0].tolist()[:-1]}")
    print(f"  - decoder_input_ids:\n    {batch['decoder_input_ids'][0].tolist()}")
    print(f"  - labels:\n    {batch['labels'][0].tolist()}")
    print(f"  - target_padding_mask:\n    {batch['target_padding_mask'][0].tolist()}")
    print("="*80)
else:
    print("Không tìm thấy file valid_token_id.jsonl để chạy thử nghiệm.")

Đang nạp dataset từ: /kaggle/working/processed_dataset/valid_token_id.jsonl
-> Nạp thành công 24,244 mẫu.

         THỬ NGHIỆM DATALOADER VÀ KIỂM TRA TENSOR ĐỘNG
1. Shape [source_ids]:         torch.Size([4, 768])
2. Shape [source_padding_mask]: torch.Size([4, 768])
3. Shape [decoder_input_ids]:  torch.Size([4, 60])
4. Shape [target_padding_mask]: torch.Size([4, 60])
5. Shape [labels]:              torch.Size([4, 60])

* Chi tiết mẫu số 1 trong batch:
  - source_ids:
    [134, 35, 4, 865, 3689, 2419, 1251, 297, 200, 10, 5372, 76, 51, 46, 572, 9, 104, 431, 1119, 4, 46, 625, 447, 128, 19, 6, 315, 301, 63, 310, 268, 400, 20, 1248, 217, 164, 25, 2798, 176, 1046, 961, 390, 4, 143, 4, 47, 50, 15, 43, 150, 288, 6175, 184, 30, 5, 2056, 27, 544, 791, 7, 860, 230, 4, 865, 3689, 2419, 11, 890, 356, 1151, 510, 1404, 4, 533, 1404, 6, 9, 472, 116, 112, 95, 31, 724, 75, 472, 116, 7, 289, 1622, 15, 43, 150, 998, 184, 30, 5, 502, 9, 1527, 4, 21, 375, 145, 107, 148, 6, 340, 636, 10, 1483, 4, 865, 938, 9